In [1]:

import re
import pandas as pd
import numpy as np
from pathlib import Path

INPUT_CSV = "/content/TwoSidesData.csv"
OUTPUT_CSV = "twosides_ddi_rag_ready.csv"
PRR_CAP = 99.0

def norm_name(s: str) -> str | None:
    if pd.isna(s):
        return None
    s = str(s)
    s = re.sub(r"\(.*?\)", " ", s)
    s = re.sub(r"[®™]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def make_title(d1, d2, cond):
    return f"{d1} + {d2} — {cond}"

def make_retrieval_text(d1, d2, cond, prr):
    prr_txt = f"{prr:.2f}" if pd.notna(prr) else "NA"
    return (
        f"When {d1} is used with {d2}, reports indicate possible "
        f"'{cond}'. Signal PRR ≈ {prr_txt}."
    )

def main():
    df = pd.read_csv(INPUT_CSV, low_memory=False)
    required = ["drug_1_concept_name","drug_2_concept_name","condition_concept_name","PRR"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    for c in ["drug_1_concept_name","drug_2_concept_name","condition_concept_name"]:
        df[c] = df[c].map(norm_name)


    df["PRR"] = pd.to_numeric(df["PRR"], errors="coerce")
    df = df.dropna(subset=["drug_1_concept_name","drug_2_concept_name","condition_concept_name","PRR"])
    df = df[df["PRR"] > 0]
    df["PRR"] = df["PRR"].clip(upper=PRR_CAP)

    pairs = np.sort(df[["drug_1_concept_name","drug_2_concept_name"]].to_numpy(), axis=1)
    df["drug1_name"] = pairs[:, 0]
    df["drug2_name"] = pairs[:, 1]

    df = (df.sort_values("PRR", ascending=False)
            .drop_duplicates(subset=["drug1_name","drug2_name","condition_concept_name"], keep="first"))

    df["pair_id"] = df["drug1_name"] + "|" + df["drug2_name"]
    df["title"] = df.apply(lambda r: make_title(r["drug1_name"], r["drug2_name"], r["condition_concept_name"]), axis=1)
    df["retrieval_text"] = df.apply(lambda r: make_retrieval_text(r["drug1_name"], r["drug2_name"], r["condition_concept_name"], r["PRR"]), axis=1)


    out_cols = [
        "title","retrieval_text",
        "drug1_name","drug2_name",
        "condition_concept_name","PRR",
        "pair_id"
    ]
    df_out = df[out_cols].rename(columns={"condition_concept_name":"condition_name"})
    Path(OUTPUT_CSV).parent.mkdir(parents=True, exist_ok=True)
    df_out.to_csv(OUTPUT_CSV, index=False)

    print(f"✅ Saved {OUTPUT_CSV}")
    print("Columns:", df_out.columns.tolist())
    print("Rows:", len(df_out))

if __name__ == "__main__":
    main()


✅ Saved twosides_ddi_rag_ready.csv
Columns: ['title', 'retrieval_text', 'drug1_name', 'drug2_name', 'condition_name', 'PRR', 'pair_id']
Rows: 501017


In [2]:

import json, random
import pandas as pd
from pathlib import Path

IN_CSV  = "twosides_ddi_rag_ready.csv"
OUT_DIR = Path("ft_out"); OUT_DIR.mkdir(exist_ok=True)

Q_TEMPLATES = [
    "What adverse event is reported when {d1} is combined with {d2}? Respond with event and PRR.",
    "Which side effect is associated with using {d1} and {d2} together? Include PRR.",
    "For the pair {d1} + {d2}, what condition has been reported? Provide PRR.",
    "List a likely risk when co-administering {d1} with {d2}. Show PRR."
]
NEG_TEMPLATES = [
    "Are there strong signals for {d1} + {d2}? If not, say 'no strong signal'.",
    "Does {d1} with {d2} show a notable side effect? If weak, say it's weak."
]

PRR_POS = 2.0
PER_POS = 2
NEG_RATIO = 0.3

def main():
    df = pd.read_csv(IN_CSV)
    req = {"drug1_name","drug2_name","condition_name","PRR","pair_id"}
    if not req.issubset(df.columns):
        raise ValueError(f"Missing columns: {req - set(df.columns)}")

    pairs = df["pair_id"].drop_duplicates().tolist()
    random.shuffle(pairs)
    n = len(pairs); n_train = int(0.8*n); n_val = int(0.1*n)
    train_pairs = set(pairs[:n_train])
    val_pairs   = set(pairs[n_train:n_train+n_val])
    test_pairs  = set(pairs[n_train+n_val:])

    splits = {"train": [], "val": [], "test": []}

    for pid, g in df.groupby("pair_id"):
        d1 = g.iloc[0]["drug1_name"]; d2 = g.iloc[0]["drug2_name"]
        pos = g[g["PRR"] >= PRR_POS]
        neg = g[g["PRR"] <  PRR_POS]


        for _, r in pos.iterrows():
            for _ in range(PER_POS):
                inst = random.choice(Q_TEMPLATES).format(d1=d1, d2=d2)
                out  = f"{r['condition_name']} (PRR ≈ {round(r['PRR'],2)})."
                ex = {"instruction": inst, "input": f"{d1} + {d2}", "output": out}
                splits["train" if pid in train_pairs else "val" if pid in val_pairs else "test"].append(ex)


        k = max(1, int(len(pos) * NEG_RATIO)) if len(pos) > 0 else min(len(neg), 1)
        if k > 0 and len(neg) > 0:
            take = neg.sample(n=min(k, len(neg)), random_state=42)
            for _, r in take.iterrows():
                inst = random.choice(NEG_TEMPLATES).format(d1=d1, d2=d2)
                qual = "no strong signal" if r["PRR"] < 1.5 else "weak signal"
                out  = f"{qual} (PRR ≈ {round(r['PRR'],2)})."
                ex = {"instruction": inst, "input": f"{d1} + {d2}", "output": out}
                splits["train" if pid in train_pairs else "val" if pid in val_pairs else "test"].append(ex)


    for name in ["train","val","test"]:
        path = OUT_DIR / f"{name}.jsonl"
        with open(path, "w", encoding="utf-8") as f:
            for ex in splits[name]:
                f.write(json.dumps(ex, ensure_ascii=False) + "\n")
        print(f"Saved {path} ({len(splits[name])} examples)")

if __name__ == "__main__":
    main()


Saved ft_out/train.jsonl (682367 examples)
Saved ft_out/val.jsonl (82483 examples)
Saved ft_out/test.jsonl (101170 examples)
